# cli

> Installed console commands: `segpt-train`, `segpt-submit`, `segpt-eval` and `segpt-predict`.

This module wires the library into two shell commands that work after `pip install -e .` or `uv sync`.

## Why two commands?

| Command | What it does |
|---|---|
| `segpt-train` | Runs the Hydra training pipeline (`scripts/train.py`) |
| `segpt-submit` | Submits a training job to a remote HPC cluster via LSF `bsub` |

Both read settings from the same `train.yaml` config file, so there is one source of truth.

## Design rule

Each function in this module does **exactly one thing**.
We develop each piece as raw code first, verify it works, then wrap it.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
#| hide
from nbdev.showdoc import *
import os

In [ ]:
#| export
from __future__ import annotations

import argparse
import runpy
import subprocess
import sys
from pathlib import Path

---
## Part 1 — `segpt-train`

### Problem: Hydra wants `--config-path` + `--config-name`, users want a plain file path

Hydra splits config location into two flags:
```
--config-path /abs/path/to/dir  --config-name train
```

We want users to just type:
```
segpt-train /any/path/to/train.yaml
```

So we intercept `sys.argv`, detect a `.yaml` argument, and rewrite it into what Hydra expects.
Any extra `key=value` overrides are passed through unchanged.

In [ ]:
# Test the argv rewrite logic in isolation — no Hydra needed

fake_argv = ['segpt-train', 'configs/train.yaml', 'train.epochs=80']
print(f'Fake argv: {fake_argv}')
print(f'Expand user: {Path(fake_argv[1]).expanduser()}')
print(f'Resolve: {Path(fake_argv[1]).expanduser().resolve()}')
cfg_file = Path(fake_argv[1]).expanduser().resolve()
cfg_file


In [ ]:

rewritten = [
    fake_argv[0],
    '--config-path', str(cfg_file.parent),
    '--config-name', cfg_file.stem,
    *fake_argv[2:],
]
print(rewritten)
# Expected: ['segpt-train', '--config-path', '<abs parent>', '--config-name', 'train', 'train.epochs=80']

In [ ]:
#| export
def _rewrite_argv_for_hydra() -> None:
    """If sys.argv[1] is a .yaml path, rewrite it to Hydra's --config-path / --config-name flags.

    Called once at the start of `segpt_train()`. Mutates `sys.argv` in place.
    """
    argv = sys.argv[1:]
    if not argv:
        return
    if argv[0].startswith("-") or not argv[0].endswith((".yml", ".yaml")):
        return  # already in Hydra format or no config supplied

    cfg_file = Path(argv[0]).expanduser().resolve()
    if not cfg_file.exists():
        raise FileNotFoundError(f"Config file not found: {cfg_file}")
    if not cfg_file.is_file():
        raise ValueError(f"Expected a config file, got a directory: {cfg_file}")

    sys.argv = [
        sys.argv[0],
        "--config-path", str(cfg_file.parent),
        "--config-name", cfg_file.stem,
        *argv[1:],
    ]

In [ ]:
path = Path('/path/to/segmentation_tools_pt')
path.exists()

In [ ]:
#| eval: false
# chdir into a machine-specific repo path — set `path` for your machine and run manually
current_path = os.chdir(
    path
)


In [ ]:
#| eval: false
# expects cwd = repo root (nbdev-test runs from nbs/) — run manually from the repo root
# Test: verify the function works with a real path on this machine
import sys
from pathlib import Path

orig = sys.argv.copy()
print(f'Current orig: {orig}')
sys.argv = ['segpt-train', str(Path('configs/train.yaml').resolve()), 'train.epochs=5']
_rewrite_argv_for_hydra()
print(f'After rewrite: {sys.argv}')
assert '--config-path' in sys.argv
assert '--config-name' in sys.argv
assert sys.argv[sys.argv.index('--config-name') + 1] == 'train'
assert 'train.epochs=5' in sys.argv
print('argv rewrite OK:', sys.argv)
sys.argv = orig  # restore


In [ ]:
sys.argv

In [ ]:
#| export
def _find_repo_root() -> Path:
    """Locate the repository root — works in both notebook and installed-module context.

    Why two strategies?
    - ``__file__`` is defined when cli.py is imported as a module -> fast, reliable.
    - ``__file__`` is NOT defined in a Jupyter notebook cell -> walk up from cwd instead.

    The cwd fallback finds the first parent that contains both ``pyproject.toml``
    and a ``scripts/`` folder, which uniquely identifies this repo.
    """
    # Strategy 1: running as an installed module
    try:
        return Path(__file__).resolve().parents[2]
    except NameError:
        pass

    # Strategy 2: running inside a Jupyter notebook
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "scripts").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root. "
        "Run this notebook from inside the cloned repository."
    )


In [ ]:
_find_repo_root()

In [ ]:
#| export

def _find_train_script() -> Path:
    """Return the absolute path to ``scripts/train.py``.

    Works in both notebook context and as an installed CLI command.
    """
    script = _find_repo_root() / "scripts" / "train.py"
    if not script.exists():
        raise FileNotFoundError(
            f"Training script not found at {script}.\n"
            "Make sure the package was installed from the repository (pip install -e .)."
        )
    return script


def _find_build_script() -> Path:
    """Return the absolute path to ``scripts/build.py``."""
    script = _find_repo_root() / "scripts" / "build.py"
    if not script.exists():
        raise FileNotFoundError(
            f"Patch-build script not found at {script}.\n"
            "Make sure the package was installed from the repository (pip install -e .)."
        )
    return script

In [ ]:
_find_train_script()

In [ ]:
#| test
# the script should exist when running from inside the repo
# (This will only pass on a machine where the package is installed editable)
try:
    p = _find_train_script()
    print('Found:', p)
except FileNotFoundError as e:
    print('Not found (expected outside repo):', e)

In [ ]:
#| export
def _run_hydra_script(script: Path, *forced_overrides: str) -> None:
    """Rewrite argv for Hydra, append forced overrides, and run a Hydra entry script."""
    _rewrite_argv_for_hydra()
    if forced_overrides:
        sys.argv = [*sys.argv, *forced_overrides]
    runpy.run_path(str(script), run_name="__main__")


def _run_train_script(*forced_overrides: str) -> None:
    """Rewrite argv for Hydra, append forced overrides, and run scripts/train.py."""
    _run_hydra_script(_find_train_script(), *forced_overrides)


def _run_build_script(*forced_overrides: str) -> None:
    """Rewrite argv for Hydra, append forced overrides, and run scripts/build.py."""
    _run_hydra_script(_find_build_script(), *forced_overrides)


def _is_build_job(config_path: Path, train_overrides: list[str], build_only_flag: bool) -> bool:
    """Return True when the submitted job should run segpt-build."""
    if build_only_flag or config_path.stem.startswith("build"):
        return True
    return any(ov.strip().lower() == "data.build_only=true" for ov in train_overrides)


def segpt_train() -> None:
    """Entry point for the `segpt-train` command.


    Accepts a plain config file path and rewrites it to Hydra format,
    then delegates to `scripts/train.py` via `runpy`.

    Example
    -------
        segpt-train configs/train.yaml
        segpt-train configs/train.yaml train.epochs=80 metrics.best_metric=recall
    """
    _run_train_script()


def segpt_build() -> None:
    """Entry point for the `segpt-build` command.


    Runs ``scripts/build.py`` with a slim build config (``configs/build.yaml``).
    ``configs/train.yaml`` still works for backward compatibility.

    Example
    -------
        segpt-build configs/build.yaml
        segpt-build configs/build.yaml data.force_rebuild=true patch.patch_size=512
    """
    _run_build_script()

In [ ]:
#| eval: false
# live demo — actually launches segpt-train; run manually
segpt_train()


---
## Part 2 — `segpt-submit`

### Problem statement

Training on a shared HPC cluster requires submitting a `bsub` job with site-specific flags.
We do not want those flags hardcoded in a shell script — they should live in the same
`train.yaml` config that drives training, under an `hpc:` section.

The `segpt-submit` command:
1. Reads `hpc:` settings from the YAML
2. Applies any `hpc.*=value` CLI overrides
3. Builds the exact `bsub` command for the chosen queue
4. Generates the inner bash job script that calls `segpt-train`
5. Submits (or prints with `--dry-run`)

### Example bsub patterns

**`gpu` queue** — shared nodes, filter by minimum GPU memory:
```bash
bsub -Is -q gpu \
     -gpu "num=3:j_exclusive=yes:gmem=24G" \
     -R 'select[ui==<your_resource_group> && osrel>=70 && type==X64LIN] rusage[mem=200G]' \
     -n 8 \
     -o logs/segpt_train_%J.out -e logs/segpt_train_%J.err
```

**`gpu_compute` queue** — dedicated big-GPU nodes (40G+), pin to specific host:
```bash
bsub -Is -q gpu_compute \
     -gpu "num=1:j_exclusive=yes" \
     -R 'select[ui==<your_resource_group> && osrel>=70 && type==X64LIN] rusage[mem=200G]' \
     -m gpu-node-01 \
     -n 8 \
     -o logs/segpt_train_%J.out -e logs/segpt_train_%J.err
```

Key differences from generic LSF:
- Memory goes in `rusage[mem=...G]` inside `-R`, **not** in `-M`
- GPU exclusivity is `j_exclusive=yes`, not `mode=exclusive_process`
- The resource `select` expression contains site-specific fields (`ui==<your_resource_group>`, `osrel`, `type`)

### Step 2a — Read `hpc:` values from YAML

Hydra config files are YAML. We use `pyyaml` if available, with a minimal regex fallback
so the CLI works even if only stdlib is installed.

In [ ]:
#| eval: false
# reads a machine-specific repo path — run manually with your checkout path
# Explore: what does pyyaml give us from train.yaml?
from pathlib import Path
import yaml

cfg_path = Path('/path/to/segmentation_tools_pt/configs/train.yaml')
data = yaml.safe_load(cfg_path.read_text())
print('hpc section:', data.get('hpc', {}))


In [ ]:
#| eval: false
# depends on `data` from the previous (gated) cell — run manually after it
# Explore: applying a CLI override like hpc.queue=gpu_compute
hpc = dict(data.get('hpc', {}))
override = 'hpc.queue=gpu_compute'
key, val = override[4:].split('=', 1)  # strip 'hpc.' prefix
hpc[key] = val
print('queue after override:', hpc['queue'])  # gpu_compute


In [ ]:
#| export
def _load_yaml(config_path: Path) -> dict:
    """Load a YAML file into a dict. Uses pyyaml if available, falls back to regex parser."""
    import re
    try:
        import yaml
        return yaml.safe_load(config_path.read_text()) or {}
    except ImportError:
        pass

    # Minimal fallback: parses top-level sections and two-space-indented key: value lines
    data: dict = {}
    section = None
    for line in config_path.read_text().splitlines():
        sec = re.match(r'^(\w+)\s*:$', line)
        kv  = re.match(r'^\s{2}(\w+)\s*:\s*(.*)$', line)
        if sec:
            section = sec.group(1)
            data.setdefault(section, {})
        elif kv and section:
            val = kv.group(2).strip().strip('"').strip("'")
            data[section][kv.group(1)] = val
    return data

In [ ]:
#| eval: false
# depends on `cfg_path` from the gated explore cell — run manually after it
_load_yaml(cfg_path)


In [ ]:
#| export
def _apply_hpc_overrides(hpc: dict, overrides: list[str]) -> dict:
    """Apply `hpc.key=value` CLI overrides to the hpc dict. Returns a new dict."""
    result = dict(hpc)
    for ov in overrides:
        if ov.startswith("hpc.") and "=" in ov:
            k, v = ov[4:].split("=", 1)
            result[k] = v
    return result

In [ ]:
#| eval: false
# reads a machine-specific repo path — run manually with your checkout path
# Test both functions together
raw = _load_yaml(Path('/path/to/segmentation_tools_pt/configs/train.yaml'))
hpc = raw.get('hpc', {})
hpc = _apply_hpc_overrides(hpc, ['hpc.queue=gpu_compute', 'hpc.host=gpu-node-01', 'train.epochs=80'])

assert hpc['queue'] == 'gpu_compute'
assert hpc['host'] == 'gpu-node-01'
assert 'train.epochs=80' not in hpc  # non-hpc overrides are NOT applied here
print('hpc after overrides:', hpc)


In [ ]:
#| export
def _parse_yaml_hpc(config_path: Path, overrides: list[str]) -> dict:
    """Load the `hpc:` section from a YAML config and apply CLI overrides.

    Example
    -------
        hpc = _parse_yaml_hpc(Path('configs/train.yaml'), ['hpc.queue=gpu_compute'])
        hpc['queue']  # 'gpu_compute'
    """
    data = _load_yaml(config_path)
    hpc  = dict(data.get("hpc", {}))
    return _apply_hpc_overrides(hpc, overrides)

### Step 2b — Build the GPU spec string

The `-gpu` flag in LSF has the form `num=N:j_exclusive=yes[:gmem=XG]`.
The `gmem` part is only added on the `gpu` queue (shared pool). On `gpu_compute` (dedicated big nodes)
you omit it because the node already has enough memory.

In [ ]:
# Explore: build GPU spec step by step
num_gpus    = 1
j_exclusive = True
gpu_mem_gb  = 24  # GB

spec = f"num={num_gpus}"
spec += ":j_exclusive=yes" if j_exclusive else ":j_exclusive=no"
if gpu_mem_gb:
    spec += f":gmem={gpu_mem_gb}G"

print(spec)  # num=3:j_exclusive=yes:gmem=24G

In [ ]:
#| export
def _build_gpu_spec(hpc: dict) -> str:
    """Build the LSF `-gpu` spec string.

    Example
    -------
        _build_gpu_spec({'num_gpus': 3, 'j_exclusive': True, 'gpu_mem_gb': 24})
        # 'num=3:j_exclusive=yes:gmem=24G'
        _build_gpu_spec({'num_gpus': 1, 'j_exclusive': True, 'gpu_mem_gb': ''})  # gpu_compute
        # 'num=1:j_exclusive=yes'
    """
    num_gpus    = hpc.get("num_gpus", 1)
    j_exclusive = str(hpc.get("j_exclusive", "true")).lower() in ("true", "yes", "1")
    gpu_mem_gb  = hpc.get("gpu_mem_gb", "")

    spec  = f"num={num_gpus}"
    spec += ":j_exclusive=yes" if j_exclusive else ":j_exclusive=no"
    if gpu_mem_gb and str(gpu_mem_gb) not in ("", "null", "None", "0"):
        spec += f":gmem={gpu_mem_gb}G"
    return spec

In [ ]:
# Test all combinations
assert _build_gpu_spec({'num_gpus': 3, 'j_exclusive': True,  'gpu_mem_gb': 24}) == 'num=3:j_exclusive=yes:gmem=24G'
assert _build_gpu_spec({'num_gpus': 1, 'j_exclusive': True,  'gpu_mem_gb': ''}) == 'num=1:j_exclusive=yes'
assert _build_gpu_spec({'num_gpus': 1, 'j_exclusive': False, 'gpu_mem_gb': 40}) == 'num=1:j_exclusive=no:gmem=40G'
print('GPU spec tests passed')

### Step 2c — Build the `-R` resource expression

In LSF, memory goes into `rusage[mem=...G]` inside the `-R` string, **not** in `-M`.
The `select[...]` part filters hosts by site-specific attributes.

In [ ]:
# Explore: build -R string
resource_select = "ui==<your_resource_group> && osrel>=70 && type==X64LIN"
mem_gb = 200
r_expr = f"select[{resource_select}] rusage[mem={mem_gb}G]"
print(r_expr)

In [ ]:
#| export
def _build_resource_expr(hpc: dict) -> str:
    """Build the LSF `-R` resource expression.

    Memory lives in `rusage[mem=...G]` (not in `-M`).
    The `select[...]` clause is site-specific and configurable via `hpc.resource_select`.

    Example
    -------
        _build_resource_expr({'mem_gb': 200})
        # "select[ui==<your_resource_group> && osrel>=70 && type==X64LIN] rusage[mem=200G]"
    """
    select = str(hpc.get(
        "resource_select",
        "ui==<your_resource_group> && osrel>=70 && type==X64LIN",
    ))
    mem_gb = hpc.get("mem_gb", 200)
    return f"select[{select}] rusage[mem={mem_gb}G]"

In [ ]:
# Test
expr = _build_resource_expr({'mem_gb': 200})
assert 'rusage[mem=200G]' in expr
assert 'your_resource_group' in expr

# Custom select
expr2 = _build_resource_expr({'mem_gb': 64, 'resource_select': 'type==X64LIN'})
assert expr2 == 'select[type==X64LIN] rusage[mem=64G]'
print('Resource expr tests passed')

### Step 2d — Assemble the full bsub command list

In [ ]:
# Explore: assemble bsub for gpu queue
hpc_gpu = {
    'job_name': 'segpt_train', 'queue': 'gpu', 'device': 'gpu',
    'interactive': True, 'num_gpus': 3, 'j_exclusive': True, 'gpu_mem_gb': 24,
    'host': '', 'num_cpus': 8, 'mem_gb': 200, 'project': ''
}

cmd = ['bsub', '-Is', '-q', 'gpu', '-n', '8',
       '-gpu', _build_gpu_spec(hpc_gpu),
       '-R', _build_resource_expr(hpc_gpu),
       '-o', 'logs/segpt_train_%J.out', '-e', 'logs/segpt_train_%J.err']
print(' '.join(f"'{c}'" if ' ' in c else c for c in cmd))

In [ ]:
#| export
def _build_bsub_cmd(hpc: dict) -> list[str]:
    """Assemble the complete bsub command argument list from the hpc config dict.

    GPU queue example:
        bsub -Is -q gpu -n 8
             -gpu 'num=3:j_exclusive=yes:gmem=24G'
             -R 'select[ui==<your_resource_group>...] rusage[mem=200G]'
             -o logs/segpt_train_%J.out -e logs/segpt_train_%J.err

    gpu_compute queue (big node, pinned host):
        bsub -Is -q gpu_compute -n 8
             -gpu 'num=1:j_exclusive=yes'
             -R 'select[...] rusage[mem=200G]'
             -m gpu-node-01
             -o logs/segpt_train_%J.out -e logs/segpt_train_%J.err
    """
    job_name    = str(hpc.get("job_name", "segpt_train"))
    queue       = str(hpc.get("queue", "gpu"))
    device      = str(hpc.get("device", "gpu")).lower()
    num_cpus    = hpc.get("num_cpus", 8)
    interactive = str(hpc.get("interactive", "true")).lower() in ("true", "yes", "1")
    host        = str(hpc.get("host", ""))
    project     = str(hpc.get("project", ""))

    cmd = ["bsub"]
    if interactive:
        cmd += ["-Is"]
    cmd += ["-q", queue, "-n", str(num_cpus)]

    if device == "gpu":
        cmd += ["-gpu", _build_gpu_spec(hpc)]

    cmd += ["-R", _build_resource_expr(hpc)]

    if host and host not in ("", "null", "None"):
        cmd += ["-m", host]
    if project and project not in ("", "null", "None"):
        cmd += ["-P", project]

    cmd += ["-o", f"logs/{job_name}_%J.out", "-e", f"logs/{job_name}_%J.err"]
    return cmd

In [ ]:
# Test: gpu queue
cmd = _build_bsub_cmd(hpc_gpu)
assert cmd[0] == 'bsub'
assert '-Is' in cmd
assert '-gpu' in cmd
assert 'num=3:j_exclusive=yes:gmem=24G' in cmd
assert '-m' not in cmd  # no host pinning on shared queue
print('GPU queue cmd:', ' '.join(cmd))

# Test: gpu_compute queue (big node, no gmem, pin host)
hpc_big = {**hpc_gpu, 'queue': 'gpu_compute', 'num_gpus': 1, 'gpu_mem_gb': '', 'host': 'gpu-node-01'}
cmd2 = _build_bsub_cmd(hpc_big)
assert 'gpu-node-01' in cmd2
assert 'gmem' not in ' '.join(cmd2)
print('gpu_compute cmd:', ' '.join(cmd2))

# Test: CPU queue (no -gpu flag)
hpc_cpu = {**hpc_gpu, 'device': 'cpu', 'queue': 'normal'}
cmd3 = _build_bsub_cmd(hpc_cpu)
assert '-gpu' not in cmd3
print('CPU cmd:', ' '.join(cmd3))

### Step 2e — Generate the inner bash job script

This is the script that LSF runs on the compute node.
It activates the environment, syncs deps, and calls `segpt-train`.

In [ ]:
# Explore: what should the job script look like?
hpc_ex = {'device': 'gpu', 'env_activate': 'source .venv/bin/activate',
           'repo_dir': '/path/to/segmentation_tools_pt'}
config_path_ex = '/path/to/segmentation_tools_pt/configs/train.yaml'
train_overrides_ex = ['train.epochs=80']

train_cmd = f'segpt-train "{config_path_ex}"'
if train_overrides_ex:
    train_cmd += ' ' + ' '.join(train_overrides_ex)

script = f"""#!/usr/bin/env bash
set -euo pipefail
echo "=== Job ${{LSB_JOBID}} | $(hostname) | $(date) ==="
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

{hpc_ex['env_activate']}

cd "{hpc_ex['repo_dir']}"
uv sync --quiet

{train_cmd}

echo "=== Done at $(date) ==="
"""
print(script)

In [ ]:
#| export
def _build_job_script(
    hpc: dict,
    config_path: Path,
    train_overrides: list[str],
    build_job: bool = False,
) -> str:
    """Generate the bash script that runs inside the LSF job.

    Minimal script: header → GPU info → cd to repo → segpt-train or segpt-build.
    No .bashrc sourcing — the LSF submission environment already has the
    correct PATH (segpt-train visible via the activated venv).
    """
    device   = str(hpc.get("device", "gpu")).lower()
    repo_dir = str(hpc.get("repo_dir", "/path/to/segmentation_tools_pt"))

    cmd_name  = "segpt-build" if build_job else "segpt-train"
    run_cmd   = f'{cmd_name} "{config_path}"'
    if train_overrides:
        run_cmd += " " + " ".join(train_overrides)

    gpu_block = (
        'nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv,noheader'
        if device == "gpu" else ""
    )

    step_label = "patch build" if build_job else "training"

    return f"""#!/usr/bin/env bash
set -euo pipefail

echo "=== Job ${{LSB_JOBID:-local}} | $(hostname) | $(date) ==="
{gpu_block}

echo "[step] entering repo..."
if [[ ! -d "{repo_dir}" ]]; then
    echo "[ERROR] repo_dir does not exist: {repo_dir}"
    echo "[ERROR] check hpc.repo_dir in your train.yaml"
    exit 1
fi
cd "{repo_dir}"
echo "[ok]   cwd=$(pwd)"
echo "[ok]   {cmd_name}=$(which {cmd_name})"

echo "[step] syncing environment..."
uv sync --quiet

echo "[step] starting {step_label}..."
echo "============================================================"
{run_cmd}
echo "============================================================"
echo "=== Done at $(date) ==="
"""


In [ ]:
# Test
script = _build_job_script(
    {'device': 'gpu', 'env_activate': 'conda activate segpt', 'repo_dir': '$HOME/repo'},
    Path('/home/user/configs/train.yaml'),
    ['train.epochs=80'],
)
assert 'nvidia-smi' in script
assert 'train.epochs=80' in script
assert 'uv sync' in script

# CPU job should NOT have nvidia-smi
script_cpu = _build_job_script({'device': 'cpu', 'env_activate': '', 'repo_dir': '$HOME/repo'},
                                Path('/home/user/configs/train.yaml'), [])
assert 'nvidia-smi' not in script_cpu
print('Job script tests passed')

### Step 2f — Print a human-readable submission summary

In [ ]:
#| export
def _print_submit_summary(
    hpc: dict,
    config_path: Path,
    train_overrides: list[str],
    *,
    build_job: bool = False,
) -> None:
    """Print a summary of what will be submitted before calling bsub."""
    device  = str(hpc.get("device", "gpu")).lower()
    project = str(hpc.get("project", ""))
    build_only = build_job

    print("=" * 50)
    print(f"  Job      : {hpc.get('job_name', 'segpt_train')}")
    print(f"  Queue    : {hpc.get('queue', 'gpu')}  |  Device: {device}")
    print(f"  Mode     : {'build-only patch cache' if build_only else 'training'}")
    if device == "gpu":
        print(f"  GPUs     : {hpc.get('num_gpus', 1)}"
              f"  j_exclusive={hpc.get('j_exclusive', True)}"
              f"  gmem={hpc.get('gpu_mem_gb', 'any')}GB")
        host = str(hpc.get("host", ""))
        if host and host not in ("", "null", "None"):
            print(f"  Host     : {host}")
    print(f"  CPUs     : {hpc.get('num_cpus', 8)}"
          f"  |  Mem: {hpc.get('mem_gb', 200)}GB")
    if project and project not in ("", "null", "None"):
        print(f"  Project  : {project}")
    print(f"  Config   : {config_path}")
    if train_overrides:
        print(f"  Overrides: {' '.join(train_overrides)}")
    print("=" * 50)

In [ ]:
# Visual check
_print_submit_summary(
    {'job_name': 'segpt_train', 'queue': 'gpu', 'device': 'gpu',
     'num_gpus': 3, 'j_exclusive': True, 'gpu_mem_gb': 24,
     'num_cpus': 8, 'mem_gb': 200, 'project': ''},
    Path('/home/user/configs/train.yaml'),
    ['train.epochs=80'],
)

### Step 2g — Assemble into `segpt_submit()`

All the pieces are tested. Now we wire them together with `argparse`.

In [ ]:
#| export
def _seggpt_main_args():
    parser = argparse.ArgumentParser(
        prog="segpt-submit",
        description=(
            "Submit a segpt-train job to a remote HPC cluster (LSF bsub).\n\n"
            "All job settings are read from the hpc: section of your train.yaml.\n"
            "Override any field on the command line with key=value pairs.\n\n"
            "hpc.* pairs  → control bsub (queue, GPUs, memory, ...)\n"
            "other pairs  → forwarded to segpt-train (epochs, batch_size, ...)"
        ),
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=(
            "examples:\n"
            "  segpt-submit configs/train.yaml\n"
            "  segpt-submit configs/train.yaml --build-only\n"
            "  segpt-submit configs/train.yaml train.epochs=80 hpc.queue=gpu_compute hpc.host=gpu-node-01\n"
            "  segpt-submit configs/train.yaml --build-only hpc.queue=gpu_compute patch.patch_size=512\n"
            "  segpt-submit configs/train.yaml hpc.device=cpu hpc.queue=normal\n"
            "  segpt-submit configs/train.yaml --dry-run\n"
        ),
    )
    parser.add_argument(
        "config",
        help="Path to your Hydra train.yaml config file",
    )
    parser.add_argument(
        "overrides",
        nargs="*",
        metavar="key=value",
        help="Override config values. hpc.* pairs control bsub; others go to segpt-train.",
    )
    parser.add_argument(
        "--build-only",
        action="store_true",
        help="Submit a patch-build-only job by forcing data.build_only=true",
    )
    parser.add_argument(
        "--dry-run",
        action="store_true",
        help="Print the bsub command and job script without submitting",
    )
    return parser.parse_args()


In [ ]:
#| export
def segpt_submit() -> None:
    """Entry point for the `segpt-submit` command.


    Reads hpc: settings from a Hydra config file and submits
    a training or patch-build job to a remote HPC cluster via LSF bsub.


    hpc.* key=value pairs control job submission.
    All other key=value pairs are forwarded to segpt-train.


    Example
    -------
        segpt-submit configs/train.yaml
        segpt-submit configs/build.yaml
        segpt-submit configs/build.yaml --build-only
        segpt-submit configs/train.yaml train.epochs=80 hpc.queue=gpu_compute hpc.host=gpu-node-01
        segpt-submit configs/build.yaml hpc.queue=normal patch.patch_size=512
        segpt-submit configs/train.yaml hpc.device=cpu hpc.queue=normal
        segpt-submit configs/train.yaml --dry-run
    """
    args = _seggpt_main_args()
    print(f"[info] parsed args: {args}", flush=True)

    config_path = Path(args.config).expanduser().resolve()
    if not config_path.exists():
        print(f"[error] Config file not found: {config_path}", file=sys.stderr)
        sys.exit(1)

    overrides: list[str]       = args.overrides
    train_overrides: list[str] = [o for o in overrides if not o.startswith("hpc.")]
    build_job = _is_build_job(config_path, train_overrides, args.build_only)

    hpc      = _parse_yaml_hpc(config_path, overrides)
    bsub_cmd = _build_bsub_cmd(hpc)
    script   = _build_job_script(hpc, config_path, train_overrides, build_job=build_job)

    _print_submit_summary(hpc, config_path, train_overrides, build_job=build_job)

    if args.dry_run:
        print("\n[dry-run] bsub command:")
        print("  " + " ".join(bsub_cmd))
        print("\n[dry-run] job script:")
        print(script)
        return

    Path("logs").mkdir(exist_ok=True)
    interactive = str(hpc.get("interactive", "true")).lower() in ("true", "yes", "1")
    if interactive:
        # -Is: bsub blocks and streams the job output live to this terminal
        proc = subprocess.run(bsub_cmd, input=script, text=True)
    else:
        # fire-and-forget: just capture the job-ID line and return
        proc = subprocess.run(bsub_cmd, input=script, text=True, capture_output=True)
        if proc.stdout:
            print(proc.stdout.strip())
        if proc.returncode != 0:
            print(proc.stderr.strip(), file=sys.stderr)
    if proc.returncode != 0:
        sys.exit(proc.returncode)


---
## Full dry-run test

Simulate `segpt-submit configs/train.yaml train.epochs=80 --dry-run` without actually calling bsub.

In [ ]:
#| eval: false
# end-to-end dry-run against a machine-specific repo path — run manually
# End-to-end dry-run simulation
import sys
from pathlib import Path

config = Path('/path/to/segmentation_tools_pt/configs/train.yaml').resolve()


In [ ]:
#| eval: false
# part of the gated dry-run chain above — run manually after cell 58

overrides = ['train.epochs=80', 'hpc.queue=gpu', 'hpc.num_gpus=1', 'hpc.gpu_mem_gb=24']

overrides


In [ ]:
#| eval: false
# part of the gated dry-run chain above — run manually after cell 58

train_overrides = [o for o in overrides if not o.startswith('hpc.')]
train_overrides


In [ ]:
#| eval: false
# part of the gated dry-run chain above — run manually after cell 58

hpc = _parse_yaml_hpc(config, overrides)
hpc


In [ ]:
#| eval: false
# part of the gated dry-run chain above — run manually after cell 58


bsub_cmd = _build_bsub_cmd(hpc)
bsub_cmd


In [ ]:
#| eval: false
# part of the gated dry-run chain above — run manually after cell 58

script = _build_job_script(hpc, config, train_overrides)
print(script)


In [ ]:
#| eval: false
# part of the gated dry-run chain above — run manually after cell 58

_print_submit_summary(hpc, config, train_overrides)
print('\nbsub command:')
print(' ', ' '.join(bsub_cmd))
print('\njob script preview (first 10 lines):')
for line in script.splitlines()[:10]:
    print(' ', line)


---
## Part 3 - `segpt-eval`

### Problem statement

After training you want to know which run is better, on one scale. `segpt-eval RUN [RUN...]` scores finished runs with the shared code in `17_eval` (`run_eval`) and prints one comparison table. Unlike training it takes **run directories**, not a Hydra config, so it is a plain `argparse` command: no Hydra, no `scripts/` entry file.

Plan:

- parse `RUN...`, `--thresholds`, `--threshold`, `--heldout`, `--objects-class`, `--out`, `--report-dir`, `--ids`, `--device`;
- call `run_eval` (imported lazily, so `segpt-train --help` does not pay for it);
- exit non-zero with a clear message when the runs cannot be compared (different val ids, missing run).

In [ ]:
#| export
def _eval_main_args(argv=None):
    parser = argparse.ArgumentParser(
        prog="segpt-eval",
        description=(
            "Evaluate finished segpt-train runs (patch mode) and print one comparison table.\n\n"
            "Each RUN is a run directory holding .hydra/config.yaml and checkpoints/best.pth\n"
            "(an RF-DETR run, model.source=rfdetr: checkpoints/last_ema.pth). U-Net and RF-DETR runs can be mixed.\n"
            "Runs whose directory names differ only by a trailing _s<seed> form one arm (mean +- spread).\n"
            "The val-id fingerprints of all runs must match, otherwise the command fails."
        ),
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=(
            "examples:\n"
            "  segpt-eval outputs/a_s42 outputs/b_s42\n"
            "  segpt-eval outputs/a_s42 outputs/a_s43 outputs/b_s42 outputs/b_s43 --heldout --out eval_out\n"
            "  segpt-eval outputs/a_s42 --thresholds 0.3 0.5 0.9 --threshold 0.5\n"
            "  segpt-eval outputs/unet_s42 outputs/rfdetr_small_s42 --heldout   # one table, one fingerprint check\n"
        ),
    )
    parser.add_argument("runs", nargs="+", metavar="RUN", help="Finished run directories")
    parser.add_argument("--thresholds", type=float, nargs="+", default=None,
                        help="Thresholds to sweep, for every run. Default: each run its own grid "
                             "(U-Net: 13 probabilities from 0.05 to 0.9999; RF-DETR: 8 scores from 0.01 to 0.7)")
    parser.add_argument("--threshold", type=float, default=0.5,
                        help="Threshold shown in the table when --heldout is not given (must be one of --thresholds)")
    parser.add_argument("--heldout", action="store_true",
                        help="Choose the threshold on one half of the source images, score the other half, both ways")
    parser.add_argument("--objects-class", default=None,
                        help="Class used for object F1 and the held-out choice (default: the last class)")
    parser.add_argument("--out", default="eval_out", help="Directory for <run>.json, <run>_per_patch.json, comparison.json")
    parser.add_argument("--report-dir", default=None, help="Build a review report here (needs --ids)")
    parser.add_argument("--ids", default=None, help="JSON list of val patch ids to put in the review report")
    parser.add_argument("--report-threshold", type=float, default=0.5, help="Threshold for the prediction shown in the report")
    parser.add_argument("--device", default=None, help="cpu or cuda (default: cuda when available)")
    args = parser.parse_args(argv)
    if args.report_dir and not args.ids:
        parser.error("--report-dir needs --ids")
    if not args.heldout and args.thresholds is not None and args.threshold not in args.thresholds:
        parser.error(f"--threshold {args.threshold} is not in --thresholds {args.thresholds}")
    return args

In [ ]:
#| export
def segpt_eval(argv=None) -> None:
    """Entry point for the `segpt-eval` command.

    Evaluate finished runs on the val split, optionally with a held-out threshold choice, and print one table.

    Example
    -------
        segpt-eval outputs/a_s42 outputs/a_s43 outputs/b_s42 --heldout --out eval_out
    """
    import json
    from segmentation_tools_pt.eval import run_eval
    args = _eval_main_args(argv)
    ids = set(json.loads(Path(args.ids).read_text())) if args.ids else None
    try:
        run_eval(args.runs, thresholds=args.thresholds, heldout=args.heldout, out=args.out, report_dir=args.report_dir,
                 report_ids=ids, report_threshold=args.report_threshold, objects_class=args.objects_class,
                 threshold=args.threshold, device=args.device)
    except (ValueError, FileNotFoundError) as e:
        print(f"[error] {e}", file=sys.stderr)
        sys.exit(1)

In [ ]:
#| test
a = _eval_main_args(['r1', 'r2', '--heldout', '--out', 'o'])
assert a.runs == ['r1', 'r2'] and a.heldout and a.out == 'o' and a.threshold == 0.5 and a.thresholds is None   # None: each run uses its own grid
a = _eval_main_args(['r1', '--thresholds', '0.3', '0.5', '--threshold', '0.3'])
assert a.thresholds == [0.3, 0.5] and a.threshold == 0.3 and not a.heldout
from segmentation_tools_pt.eval import DEFAULT_THRESHOLDS
from segmentation_tools_pt.rfdetr import RFDETR_THRESHOLDS
assert 0.5 in DEFAULT_THRESHOLDS and 0.5 in RFDETR_THRESHOLDS      # the default --threshold is swept by both kinds of run
for bad in (['r1', '--thresholds', '0.3', '--threshold', '0.5'], ['r1', '--report-dir', 'x'], []):
    try:
        _eval_main_args(bad)
    except SystemExit as e:
        assert e.code == 2
    else:
        raise AssertionError(f'{bad} should be rejected')

In [ ]:
#| test
# a path that is not a run fails loudly with exit code 1, not a traceback
import contextlib, io, tempfile
err = io.StringIO()
with tempfile.TemporaryDirectory() as d, contextlib.redirect_stderr(err):
    try:
        segpt_eval([d, '--out', d + '/o'])
    except SystemExit as e:
        assert e.code == 1
    else:
        raise AssertionError('expected exit 1')
assert 'not a finished segpt-train run' in err.getvalue(), err.getvalue()

In [ ]:
#| export
def _run_repo_script(name: str) -> None:
    """Run `scripts/<name>.py` from the repo as `__main__`, with `scripts/` importable (they share `_bench_common`)."""
    script = _find_repo_root() / "scripts" / f"{name}.py"
    if not script.exists():
        raise FileNotFoundError(f"Script not found at {script}. Install the package from the repository (uv sync).")
    sys.path.insert(0, str(script.parent))
    sys.argv[0] = f"{name}.py"
    runpy.run_path(str(script), run_name="__main__")


def segpt_sweep() -> None:
    """Entry point for `segpt-sweep`: score one finished semantic run at several probability thresholds.

    Example
    -------
        segpt-sweep outputs/run_s42 --out sweep.json
    """
    _run_repo_script("semantic_threshold_sweep")


def segpt_heldout() -> None:
    """Entry point for `segpt-heldout`: held-out threshold choice and the Go/No-go rule over sweep outputs (see `segpt-eval --heldout`)."""
    _run_repo_script("heldout_compare")


def segpt_rfdetr_benchmark() -> None:
    """Entry point for `segpt-rfdetr-benchmark`: the #102 RF-DETR train/eval benchmark (needs the `rfdetr` extra).

    Example
    -------
        segpt-rfdetr-benchmark eval --cache PATCHES --out RUN
    """
    _run_repo_script("rfdetr_benchmark")

In [ ]:
#| test
# every script the wrappers point at exists in this checkout
root = _find_repo_root()
for n in ("semantic_threshold_sweep", "heldout_compare", "rfdetr_benchmark"):
    assert (root / "scripts" / f"{n}.py").exists(), n
for f in (segpt_sweep, segpt_heldout, segpt_rfdetr_benchmark):
    assert callable(f)

---
## Part 4 - `segpt-predict`

### Problem statement

A finished run (U-Net or RF-DETR) should predict on new images with one command. Until now an agent wrote Python around
`predict_tiled` / `predict_tiled_rfdetr` and had to remember the differences between the two model families.

### Plan

1. Read the run's saved `.hydra/config.yaml`: it says which model family, how many classes, which channel mode, the crop size.
2. Collect the input images (files or folders) and turn each into the model's input channels. Wrong channel counts and 16-bit images fail with a message naming the file.
3. Load the right model: U-Net `checkpoints/best.pth` (probabilities, strict `>`), RF-DETR `checkpoints/last_ema.pth` (confidences, inclusive `>=`), tiled at the training crop size.
4. Write one mask per image (and class), one overlay per image and one `summary.json`.

Layout of the output folder (fixed, tested):

```
OUT/masks/<id>.png            0/255 mask (one class); several classes: <id>_<class>.png
OUT/overlays/<id>.png         the first input channel with the masks tinted
OUT/summary.json              run, checkpoint, threshold, tile size, per-image files and foreground fraction
```

We first build a tiny finished run on synthetic data, so every step can be checked on CPU in a second.

In [ ]:
# Experiment: a tiny finished U-Net run on synthetic data (no training: random weights are enough to test the plumbing)
import json, tempfile
import cv2, numpy as np, torch
from omegaconf import OmegaConf

def _make_run(root, family='unet', n_source=1, crop=32, classes=('defect',)):
    "A fake finished run under `root`: saved config, patch stats and (U-Net) a checkpoint."
    from segmentation_tools_pt.model import get_model_from_cfg
    root = Path(root); run = root / 'run_s42'
    mode = {1: 'single', 2: 'two_channel'}[n_source]
    in_ch = {1: 1, 2: 3}[n_source]
    cfg = OmegaConf.create({
        'data': {'data_mode': mode, 'patch_dir': str(root / 'patches'), 'class_names': list(classes)},
        'patch': {'patch_size': crop}, 'train': {'num_classes': len(classes), 'crop_size': crop},
        'model': {'source': 'rfdetr' if family == 'rfdetr' else 'custom_unet', 'arch': 'small',
                  'custom_unet': {'encoder_depth': 2, 'base_channels': 8, 'num_groups': 4}},
        'inference': {'tile_size': crop, 'overlap': crop // 4, 'inference_batch_size': 4}})
    (run / '.hydra').mkdir(parents=True); (run / 'checkpoints').mkdir()
    OmegaConf.save(cfg, run / '.hydra' / 'config.yaml')
    (root / 'patches').mkdir()
    (root / 'patches' / 'stats.json').write_text(json.dumps({'mean': [0.3] * in_ch, 'std': [0.2] * in_ch}))
    if family == 'unet':
        torch.manual_seed(0)
        model = get_model_from_cfg(cfg, in_ch, crop_size=crop, pretrained=False)
        torch.save({'model_state': model.state_dict()}, run / 'checkpoints' / 'best.pth')
    else:
        (run / 'checkpoints' / 'last_ema.pth').write_bytes(b'stub')
    return run

def _make_images(folder, n=2, size=(70, 90), n_source=1):
    "Noisy grey images with a bright square, as `<id>.png` (or `<id>_ch0.png` + `<id>_ch1.png`)."
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    for i in range(n):
        for c in range(n_source):
            img = rng.integers(20, 80, size, dtype=np.uint8); img[20:40, 30:50] = 220
            name = f'img{i}.png' if n_source == 1 else f'img{i}_ch{c}.png'
            cv2.imwrite(str(folder / name), img)
    return folder

tmp = Path(tempfile.mkdtemp())
run = _make_run(tmp)
imgs = _make_images(tmp / 'imgs')
sorted(p.name for p in run.rglob('*') if p.is_file()), sorted(p.name for p in imgs.iterdir())

Verified on the fake run: the config, stats and checkpoint are all the command needs. Now the pieces, each exported once its behaviour is pinned down.

### Step 4a - read the run and the input images

The run config is the single source for family, channels and crop size. Inputs are checked before any model is loaded so a bad image fails fast.

In [ ]:
#| export
import json
import numpy as np

PREDICT_THRESHOLDS = {'unet': 0.5, 'rfdetr': 0.2}   # RF-DETR scores are confidences, not probabilities: its useful range sits lower
_IMAGE_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')


class PredictError(Exception):
    "An expected `segpt-predict` failure; the message names the file or config key to fix."


def _run_config(run_dir):
    from omegaconf import OmegaConf
    p = Path(run_dir) / '.hydra' / 'config.yaml'
    if not p.exists():
        raise PredictError(f'{p} not found: {run_dir} is not a finished segpt-train run')
    return OmegaConf.load(p)


def _collect_inputs(paths, n_source):
    """`[(sample_id, [channel files])]` for image files and folders; several-channel runs find `<id>_ch<i>` / `<id>_image<i+1>` siblings."""
    from segmentation_tools_pt.dataset_build import _stem_to_sample_id
    files = []
    for p in map(Path, paths):
        if p.is_dir():
            files += sorted(q for q in p.iterdir() if q.suffix.lower() in _IMAGE_EXTS)
        elif p.is_file():
            files.append(p)
        else:
            raise PredictError(f'{p} not found')
    if not files:
        raise PredictError(f'no images ({", ".join(_IMAGE_EXTS)}) found in {list(map(str, paths))}')
    if n_source == 1:
        return [(f.stem, [f]) for f in files]
    groups = {}
    for f in files:
        groups.setdefault((f.parent, _stem_to_sample_id(f.stem, n_source), f.suffix), None)
    out = []
    for (folder, sid, ext) in groups:
        chans = []
        for i in range(n_source):
            found = next((c for c in (folder / f'{sid}_ch{i}{ext}', folder / f'{sid}_image{i + 1}{ext}') if c.exists()), None)
            if found is None:
                raise PredictError(f'{folder / f"{sid}_ch{i}{ext}"} not found: this run needs {n_source} source channels per image')
            chans.append(found)
        out.append((sid, chans))
    return out


def _read_input(paths, preset, mode):
    "The `(H, W, C)` uint8 model input of one sample: its channel files read, checked and combined like at training time."
    import cv2
    from segmentation_tools_pt.channels import build_image
    arrs = []
    for p in paths:
        a = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
        if a is None:
            raise PredictError(f'{p} cannot be read as an image')
        if a.ndim == 3 and a.shape[2] == 1:
            a = a[..., 0]
        if a.ndim != 2:
            raise PredictError(f'{p} has {a.shape[2]} channels but this run expects one channel per file '
                               f'(data.data_mode={mode}: {preset["n_source_channels"]} source file(s) per image)')
        if a.dtype != np.uint8:
            raise PredictError(f'{p} is {a.dtype}: only 8-bit images are supported (the training patches are 8-bit); convert it first')
        arrs.append(a)
    if len({a.shape for a in arrs}) != 1:
        raise PredictError(f'channel files {[str(p) for p in paths]} differ in size')
    return build_image(arrs, preset['channel_names'])

In [ ]:
#| test
run_cfg = _run_config(run)
assert run_cfg.train.crop_size == 32
assert _collect_inputs([imgs], 1)[0][0] == 'img0'
assert len(_collect_inputs([imgs, imgs / 'img0.png'], 1)) == 3

### Step 4b - load the model and a `score(image, threshold)` function

Both families end in the same shape, `(K, H, W)` booleans. U-Net goes through `predict_tiled` (sigmoid, strict `>`); RF-DETR through `predict_tiled_rfdetr` at the training crop size, where `> 0` of the blended map means "some tile detected it at or above the threshold".

In [ ]:
#| export
def _load_scorer(cfg, run_dir, preset, device, loader=None):
    """`(family, checkpoint, score)` for a run; `score(image, threshold)` gives `(K, H, W)` bool masks of an `(H, W, C)` uint8 image."""
    import torch
    from segmentation_tools_pt.hydra_pipeline import resolve_crop_size, resolve_inference_cfg, resolve_mask_cfg, resolve_task
    from segmentation_tools_pt.inference import predict_tiled
    from segmentation_tools_pt.rfdetr import is_rfdetr_cfg, load_rfdetr_predictor, predict_tiled_rfdetr, rfdetr_checkpoint
    crop = resolve_crop_size(cfg)
    inf = resolve_inference_cfg(cfg)
    knobs = dict(tile_size=crop, overlap=inf['overlap'], inference_batch_size=inf['inference_batch_size'])
    if is_rfdetr_cfg(cfg):
        ckpt = rfdetr_checkpoint(run_dir)
        names = resolve_mask_cfg(cfg)['class_names']
        try:
            predictor = load_rfdetr_predictor(run_dir, num_classes=len(names) if names else 1, loader=loader)
        except ImportError as e:
            raise PredictError(str(e)) from e
        return 'rfdetr', ckpt, lambda img, thr: predict_tiled_rfdetr(predictor, img.transpose(2, 0, 1), device, thr, **knobs).cpu().numpy() > 0
    import albumentations as A
    from segmentation_tools_pt.model import get_model_from_cfg
    ckpt = Path(run_dir) / 'checkpoints' / 'best.pth'
    stats = Path(cfg.data.patch_dir) / 'stats.json'
    for p, what in ((ckpt, 'a finished U-Net run has checkpoints/best.pth'), (stats, 'it holds the training mean/std the model was fed')):
        if not p.exists():
            raise PredictError(f'{p} not found: {what}')
    s = json.loads(stats.read_text())
    norm = A.Normalize(mean=tuple(s['mean']), std=tuple(s['std']), max_pixel_value=255.0)
    model = get_model_from_cfg(cfg, preset['in_channels'], crop_size=crop, pretrained=False)
    model.load_state_dict(torch.load(ckpt, map_location='cpu', weights_only=True)['model_state'])
    model.to(device).eval()
    task = resolve_task(cfg)

    def score(img, thr):
        x = torch.from_numpy(norm(image=img)['image'].transpose(2, 0, 1).copy()).float()
        prob = predict_tiled(model, x, torch.device(device), task=task, **knobs).cpu()
        if task == 'multiclass':
            return torch.stack([prob.argmax(0) == k for k in range(prob.shape[0])]).numpy()
        return (prob > thr).numpy()
    return 'unet', ckpt, score

### Step 4c - write masks, overlays and the summary

In [ ]:
#| export
_OVERLAY_COLORS = ((255, 0, 0), (0, 200, 0), (0, 128, 255), (255, 200, 0), (200, 0, 255))   # RGB, cycled per class


def _overlay(img, masks, alpha=0.5):
    "The first input channel as RGB with each class mask tinted in its own colour."
    base = np.repeat(img[..., :1], 3, axis=2).astype(np.float32)
    for k, m in enumerate(masks):
        col = np.array(_OVERLAY_COLORS[k % len(_OVERLAY_COLORS)], np.float32)
        base[m] = (1 - alpha) * base[m] + alpha * col
    return base.astype(np.uint8)


def predict_run(run_dir, images, out='predict_out', threshold=None, device=None, loader=None) -> dict:
    """Predict `images` (files or folders) with a finished run and write masks, overlays and `summary.json` under `out`.

    Reads the run's `.hydra/config.yaml` for the model family, crop size, class names and channel mode. `threshold=None`
    uses `PREDICT_THRESHOLDS` of the family. `loader(ckpt)` replaces the RF-DETR model loader (tests). Returns the summary dict.
    Raises `PredictError` (or `FileNotFoundError`) naming the file or key when the run or an image cannot be used.
    """
    import cv2, torch
    from segmentation_tools_pt.hydra_pipeline import resolve_channel_preset, resolve_mask_cfg
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    cfg = _run_config(run_dir)
    mode = cfg.data.get('data_mode', 'single')
    preset = resolve_channel_preset(mode)
    samples = _collect_inputs(images, preset['n_source_channels'])
    family, ckpt, score = _load_scorer(cfg, run_dir, preset, device, loader)
    thr = PREDICT_THRESHOLDS[family] if threshold is None else float(threshold)
    names = resolve_mask_cfg(cfg)['class_names']
    out = Path(out); (out / 'masks').mkdir(parents=True, exist_ok=True); (out / 'overlays').mkdir(exist_ok=True)
    rows = []
    for sid, paths in samples:
        img = _read_input(paths, preset, mode)
        masks = score(img, thr)
        cls = list(names) if names and len(names) == len(masks) else [f'class{k}' for k in range(len(masks))]
        files = []
        for k, m in enumerate(masks):
            f = out / 'masks' / (f'{sid}.png' if len(masks) == 1 else f'{sid}_{cls[k]}.png')
            cv2.imwrite(str(f), m.astype(np.uint8) * 255)
            files.append(str(f.relative_to(out)))
        cv2.imwrite(str(out / 'overlays' / f'{sid}.png'), _overlay(img, masks)[..., ::-1])
        rows.append({'id': sid, 'inputs': [str(p) for p in paths], 'shape': list(img.shape[:2]), 'masks': files,
                     'overlay': f'overlays/{sid}.png', 'foreground_fraction': {c: float(m.mean()) for c, m in zip(cls, masks)}})
    summary = {'run': str(Path(run_dir).resolve()), 'family': family, 'checkpoint': str(ckpt), 'threshold': thr, 'device': device,
               'tile_size': int(cfg.train.get('crop_size') or cfg.patch.patch_size), 'images': rows}
    (out / 'summary.json').write_text(json.dumps(summary, indent=2))
    return summary

In [ ]:
#| test
# explore result 1: a U-Net run predicts, files land where the layout says
s = predict_run(run, [imgs], out=tmp / 'out', device='cpu')
out = tmp / 'out'
assert sorted(p.name for p in (out / 'masks').iterdir()) == ['img0.png', 'img1.png'], list((out / 'masks').iterdir())
assert sorted(p.name for p in (out / 'overlays').iterdir()) == ['img0.png', 'img1.png']
m = cv2.imread(str(out / 'masks' / 'img0.png'), cv2.IMREAD_UNCHANGED)
assert m.shape == (70, 90) and set(np.unique(m)) <= {0, 255}
o = cv2.imread(str(out / 'overlays' / 'img0.png'), cv2.IMREAD_UNCHANGED)
assert o.shape == (70, 90, 3)
sm = json.loads((out / 'summary.json').read_text())
assert sm['family'] == 'unet' and sm['threshold'] == 0.5 and sm['tile_size'] == 32 and len(sm['images']) == 2
assert sm['checkpoint'].endswith('checkpoints/best.pth')

In [ ]:
#| test
# the threshold override changes the mask: probabilities are in (0, 1), so 0.0 keeps (nearly) all pixels, 1.0 none
predict_run(run, [imgs / 'img0.png'], out=tmp / 'lo', threshold=0.0, device='cpu')
predict_run(run, [imgs / 'img0.png'], out=tmp / 'hi', threshold=1.0, device='cpu')
lo = cv2.imread(str(tmp / 'lo' / 'masks' / 'img0.png'), cv2.IMREAD_UNCHANGED); hi = cv2.imread(str(tmp / 'hi' / 'masks' / 'img0.png'), cv2.IMREAD_UNCHANGED)
assert (lo > 0).mean() > 0.99 and (hi > 0).sum() == 0

### Step 4d - the command line

Expected failures print `[error] ...` naming the file and exit 1, like `segpt-eval`.

In [ ]:
#| export
def _predict_main_args(argv=None):
    parser = argparse.ArgumentParser(
        prog="segpt-predict",
        description=(
            "Predict on new images with a finished segpt-train run (U-Net or RF-DETR) and write masks, overlays and a summary.\n\n"
            "RUN holds .hydra/config.yaml and checkpoints/best.pth (U-Net) or checkpoints/last_ema.pth (RF-DETR, model.source=rfdetr).\n"
            "The saved config decides the model, tile size (= training crop size), class names and channel mode.\n"
            "IMAGES are 8-bit image files or folders of them; several-channel runs find <id>_ch0, <id>_ch1, ... next to each other.\n\n"
            "output layout:\n"
            "  OUT/masks/<id>.png        0/255 mask (several classes: <id>_<class>.png)\n"
            "  OUT/overlays/<id>.png     first input channel with the masks tinted\n"
            "  OUT/summary.json          run, checkpoint, threshold, tile size, files and foreground fraction per image\n\n"
            "threshold: U-Net probability, kept where  p > t   (default 0.5).\n"
            "           RF-DETR confidence score, kept where  score >= t   (default 0.2). It is a detector confidence, not a probability."
        ),
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=(
            "examples:\n"
            "  segpt-predict outputs/unet_s42 new_images/ --out predictions\n"
            "  segpt-predict outputs/unet_s42 a.png b.png --threshold 0.7 --device cpu\n"
            "  segpt-predict outputs/rfdetr_small_s42 new_images/        # tiles at the training crop size automatically\n"
        ),
    )
    parser.add_argument("run", metavar="RUN", help="Finished run directory")
    parser.add_argument("images", nargs="+", metavar="IMAGES", help="Image files or folders")
    parser.add_argument("--out", default="predict_out", help="Output folder (default: predict_out)")
    parser.add_argument("--threshold", type=float, default=None, help="Override the default threshold of the run's model family")
    parser.add_argument("--device", default=None, help="cpu or cuda (default: cuda when available)")
    return parser.parse_args(argv)


def segpt_predict(argv=None) -> None:
    """Entry point for the `segpt-predict` command.

    Predict on new images with a finished run; expected failures print `[error] <message>` and exit 1.

    Example
    -------
        segpt-predict outputs/unet_s42 new_images/ --out predictions
    """
    args = _predict_main_args(argv)
    try:
        s = predict_run(args.run, args.images, out=args.out, threshold=args.threshold, device=args.device)
    except (PredictError, ValueError, FileNotFoundError) as e:
        print(f"[error] {e}", file=sys.stderr)
        sys.exit(1)
    print(f"{len(s['images'])} image(s), {s['family']} threshold {s['threshold']} -> {args.out}/summary.json")

In [ ]:
#| test
# expected failures exit 1 with the file named, no traceback
import contextlib, io
def _fails(argv):
    err = io.StringIO()
    with contextlib.redirect_stderr(err):
        try: segpt_predict(argv)
        except SystemExit as e: code = e.code
        else: code = 0
    return code, err.getvalue()

(run / 'checkpoints' / 'best.pth').rename(run / 'checkpoints' / 'gone.pth')
code, msg = _fails([str(run), str(imgs), '--out', str(tmp / 'e1'), '--device', 'cpu'])
assert code == 1 and 'checkpoints/best.pth' in msg and 'Traceback' not in msg, msg
(run / 'checkpoints' / 'gone.pth').rename(run / 'checkpoints' / 'best.pth')

code, msg = _fails([str(tmp / 'nope'), str(imgs), '--out', str(tmp / 'e2')])
assert code == 1 and '.hydra/config.yaml' in msg, msg

rgb = tmp / 'rgb'; rgb.mkdir(); cv2.imwrite(str(rgb / 'color.png'), np.zeros((40, 40, 3), np.uint8))
code, msg = _fails([str(run), str(rgb), '--out', str(tmp / 'e3'), '--device', 'cpu'])
assert code == 1 and 'color.png' in msg and '3 channels' in msg, msg

wide = tmp / 'wide'; wide.mkdir(); cv2.imwrite(str(wide / 'deep.png'), np.zeros((40, 40), np.uint16))
code, msg = _fails([str(run), str(wide), '--out', str(tmp / 'e4'), '--device', 'cpu'])
assert code == 1 and 'deep.png' in msg and 'uint16' in msg, msg

code, msg = _fails([str(run), str(tmp / 'empty_dir')])
assert code == 1 and 'empty_dir' in msg, msg

# the happy path through the console entry point
segpt_predict([str(run), str(imgs), '--out', str(tmp / 'cli'), '--device', 'cpu'])
assert (tmp / 'cli' / 'summary.json').exists()

In [ ]:
#| test
# several classes, several source channels: one mask per class, channel files grouped by id
run2 = _make_run(tmp / 'multi', n_source=2, classes=('chip', 'void'))
imgs2 = _make_images(tmp / 'multi' / 'imgs', n=1, n_source=2)
s2 = predict_run(run2, [imgs2], out=tmp / 'multi' / 'out', device='cpu')
assert sorted(p.name for p in (tmp / 'multi' / 'out' / 'masks').iterdir()) == ['img0_chip.png', 'img0_void.png']
assert len(s2['images']) == 1 and s2['images'][0]['inputs'][1].endswith('img0_ch1.png')
(imgs2 / 'img0_ch1.png').unlink()
try: predict_run(run2, [imgs2 / 'img0_ch0.png'], out=tmp / 'multi' / 'out2', device='cpu')
except PredictError as e: assert 'img0_ch1.png' in str(e), e
else: raise AssertionError('missing channel file not reported')

In [ ]:
#| test
# RF-DETR path with a stub model: confidence threshold is inclusive, tiles are cut at the training crop size, 0-255 input
run3 = _make_run(tmp / 'detr', family='rfdetr')
seen = []
class _Det:
    "supervision.Detections-like: one 10x10 instance mask, confidence 0.4, kept when it reaches the threshold"
    def __init__(self, thr, size):
        n = int(0.4 >= thr)
        self.mask = np.zeros((n, size[1], size[0]), bool); self.mask[:, 2:12, 2:12] = True
        self.confidence = np.full(n, 0.4); self.class_id = np.zeros(n, int)
class _Stub:
    def predict(self, image, threshold=0.0):
        seen.append((image.size, threshold)); return _Det(threshold, image.size)
s3 = predict_run(run3, [imgs / 'img0.png'], out=tmp / 'detr' / 'out', loader=lambda ckpt: _Stub(), device='cpu')
assert s3['family'] == 'rfdetr' and s3['threshold'] == 0.2 and s3['checkpoint'].endswith('last_ema.pth')
assert all(size == (32, 32) for size, _ in seen), seen          # tiles at the training crop size
fg = s3['images'][0]['foreground_fraction']['defect']
assert fg > 0
predict_run(run3, [imgs / 'img0.png'], out=tmp / 'detr' / 'out_hi', threshold=0.5, loader=lambda ckpt: _Stub(), device='cpu')
hi = cv2.imread(str(tmp / 'detr' / 'out_hi' / 'masks' / 'img0.png'), cv2.IMREAD_UNCHANGED)
assert hi.sum() == 0   # 0.4 < 0.5: nothing kept
predict_run(run3, [imgs / 'img0.png'], out=tmp / 'detr' / 'out_eq', threshold=0.4, loader=lambda ckpt: _Stub(), device='cpu')
assert cv2.imread(str(tmp / 'detr' / 'out_eq' / 'masks' / 'img0.png'), cv2.IMREAD_UNCHANGED).sum() > 0   # score == threshold is kept
# no checkpoint: names the file
(run3 / 'checkpoints' / 'last_ema.pth').unlink()
try: predict_run(run3, [imgs], out=tmp / 'detr' / 'e', loader=lambda ckpt: _Stub())
except FileNotFoundError as e: assert 'last_ema.pth' in str(e)
else: raise AssertionError

---
## Part 5 - `segpt-run`

### Problem statement

Build, train, evaluate and predict are four commands. Each is started and checked by hand, and when training crashes nothing says where to resume.

### Plan

1. A small run file names the data, the training config, the arms (model choices), the seeds, the images to predict and optional extra arguments.
2. From it we make a **plan**: stages `build`, `train-<arm>_s<seed>`, `eval`, `predict-<arm>_s<seed>`, each with the command it runs. `--dry-run` prints the plan and stops.
3. Every stage runs the existing command in its own process (so behaviour equals running it by hand) and writes a marker `<run_dir>/.segpt_run/<stage>.done` when it succeeds.
4. A re-run skips stages with a marker. A failed stage leaves no marker, so the next run resumes there (training also resumes from its `latest.pth`). `--force STAGE` re-runs only that stage.

Run directory layout (fixed, tested):

```
RUN/build/            hydra run dir of segpt-build         (patch cache: RUN/patches unless data.patch_dir is set)
RUN/train/<arm>_s<seed>/    one segpt-train run per arm and seed
RUN/eval/             segpt-eval output (comparison.json, per-run json)
RUN/predict/<arm>_s<seed>/  segpt-predict output
RUN/.segpt_run/       stage markers
```

No ledger, no acceptance gate, no job submission: it is a small runner that calls the commands we already have.

In [ ]:
# Experiment: a run file and the plan we expect from it (nothing is run yet)
import yaml, tempfile
_tmp = Path(tempfile.mkdtemp())
_cfg = {'run_dir': str(_tmp / 'run'), 'config': 'configs/train.yaml',
        'overrides': ['data.raw_dir=/data/raw', 'train.epochs=2'],
        'arms': {'unet': ['model.source=custom_unet'], 'rf': ['model.source=rfdetr']},
        'seeds': [42, 43], 'predict': {'images': ['/data/new']}}
print(yaml.safe_dump(_cfg, sort_keys=False))
print('expected stages: build, train-unet_s42, train-unet_s43, train-rf_s42, train-rf_s43, eval, predict-... x4')

In [ ]:
#| export
import json, re, shutil
import yaml

RUN_FIELDS = {'run_dir', 'config', 'build_config', 'overrides', 'arms', 'seeds', 'eval', 'predict'}
_SEED_SUFFIX = re.compile(r'_s\d+$')

class RunError(Exception):
    "An expected `segpt-run` failure (bad run file, failed stage): printed without a traceback, exit 1."

def _load_run_file(path, extra_overrides=()):
    "Read a run file into a plain dict; paths are made absolute against the file's folder."
    path = Path(path)
    if not path.is_file(): raise RunError(f'run file not found: {path}')
    raw = yaml.safe_load(path.read_text()) or {}
    unknown = set(raw) - RUN_FIELDS
    if unknown: raise RunError(f'{path}: unknown key(s) {sorted(unknown)}; allowed: {sorted(RUN_FIELDS)}')
    for key in ('run_dir', 'config'):
        if not raw.get(key): raise RunError(f"{path}: '{key}' is required")
    here = path.resolve().parent
    def absolute(p): return str((here / str(p)).resolve()) if not Path(str(p)).is_absolute() else str(p)
    arms = raw.get('arms') or {'run': []}
    for name in arms:
        if _SEED_SUFFIX.search(str(name)): raise RunError(f"{path}: arm name '{name}' must not end in _s<number> (the seed is added)")
    pred = dict(raw.get('predict') or {})
    return {
        'file': str(path.resolve()), 'run_dir': absolute(raw['run_dir']), 'config': absolute(raw['config']),
        'build_config': absolute(raw.get('build_config') or raw['config']),
        'overrides': [str(o) for o in (raw.get('overrides') or [])] + list(extra_overrides),
        'arms': {str(k): [str(o) for o in (v or [])] for k, v in arms.items()},
        'seeds': [int(s) for s in (raw.get('seeds') or [42])],
        'eval': {'args': [str(a) for a in (raw.get('eval') or {}).get('args', [])]},
        'predict': {'images': [absolute(i) for i in pred.get('images', [])], 'args': [str(a) for a in pred.get('args', [])]},
    }

In [ ]:
#| export
_STAGE_COMMANDS = {'segpt-build': 'segpt_build', 'segpt-train': 'segpt_train', 'segpt-eval': 'segpt_eval', 'segpt-predict': 'segpt_predict'}

def plan_run(cfg) -> list:
    "The ordered stages of a run: dicts with `id`, `stage`, `command`, `args` (the CLI call) and `dir` (what the stage owns)."
    run = Path(cfg['run_dir'])
    shared = list(cfg['overrides'])
    if not any(o.startswith('data.patch_dir=') for o in shared): shared.append(f'data.patch_dir={run / "patches"}')
    stages = [{'id': 'build', 'stage': 'build', 'command': 'segpt-build', 'dir': run / 'build',
               'args': [cfg['build_config'], *shared, f'hydra.run.dir={run / "build"}']}]
    runs = []
    for arm, arm_ovr in cfg['arms'].items():
        for seed in cfg['seeds']:
            name = f'{arm}_s{seed}'; d = run / 'train' / name; runs.append((name, d))
            args = [cfg['config'], *shared, *arm_ovr, f'train.seed={seed}', f'hydra.run.dir={d}']
            if (d / 'checkpoints' / 'latest.pth').exists(): args.append('train.resume=true')
            stages.append({'id': f'train-{name}', 'stage': 'train', 'command': 'segpt-train', 'dir': d, 'args': args})
    stages.append({'id': 'eval', 'stage': 'eval', 'command': 'segpt-eval', 'dir': run / 'eval',
                   'args': [*[str(d) for _, d in runs], '--out', str(run / 'eval'), *cfg['eval']['args']]})
    if cfg['predict']['images']:
        for name, d in runs:
            out = run / 'predict' / name
            stages.append({'id': f'predict-{name}', 'stage': 'predict', 'command': 'segpt-predict', 'dir': out,
                           'args': [str(d), *cfg['predict']['images'], '--out', str(out), *cfg['predict']['args']]})
    return stages

def _marker(cfg, stage): return Path(cfg['run_dir']) / '.segpt_run' / f"{stage['id']}.done"

def _forced(stage, force):
    return stage['id'] in force or stage['stage'] in force

In [ ]:
#| export
def _subprocess_exec(stage):
    "Run one stage's existing CLI in its own process; the return code is the stage's result."
    fn = _STAGE_COMMANDS[stage['command']]
    code = f'from segmentation_tools_pt.cli import {fn}; {fn}()'
    return subprocess.run([sys.executable, '-c', code, *stage['args']]).returncode

def _show(stage, state):
    print(f"[{state:7}] {stage['id']:<28} {stage['command']} {' '.join(stage['args'])}")

def run_pipeline(cfg, dry_run=False, force=(), execute=None) -> dict:
    "Run the plan of `cfg` stage by stage; returns `{stage id: 'done' | 'skipped' | 'planned'}`, raises `RunError` at the first failed stage."
    execute = execute or _subprocess_exec
    force = set(force)
    stages = plan_run(cfg)
    known = {s['id'] for s in stages} | {s['stage'] for s in stages}
    bad = force - known
    if bad: raise RunError(f'--force {sorted(bad)}: not a stage of this run; stages are {sorted(known)}')
    result = {}
    if not dry_run:
        Path(cfg['run_dir']).mkdir(parents=True, exist_ok=True)
        shutil.copy(cfg['file'], Path(cfg['run_dir']) / 'run.yaml')
    for st in stages:
        marker, forced = _marker(cfg, st), _forced(st, force)
        todo = forced or not marker.exists()
        if dry_run:
            _show(st, 'forced' if forced else 'todo' if todo else 'done'); result[st['id']] = 'planned' if todo else 'skipped'; continue
        if not todo:
            _show(st, 'skipped'); result[st['id']] = 'skipped'; continue
        if forced:
            marker.unlink(missing_ok=True)
            if st['stage'] != 'build': shutil.rmtree(st['dir'], ignore_errors=True)
            else: st = {**st, 'args': [*st['args'], 'data.force_rebuild=true']}
        _show(st, 'running')
        rc = execute(st)
        if rc != 0:
            raise RunError(f"stage '{st['id']}' failed (exit {rc}); fix it and run the same command again: finished stages are skipped and this one resumes")
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps({'stage': st['id'], 'command': st['command'], 'args': st['args']}, indent=1))
        result[st['id']] = 'done'
    return result

In [ ]:
#| export
def _run_main_args(argv=None):
    parser = argparse.ArgumentParser(
        prog="segpt-run",
        description=(
            "Run build, train (per arm and seed), eval and predict as stages in one run directory.\n\n"
            "CONFIG is a run file (YAML): run_dir, config (a segpt-train config), optional build_config, overrides,\n"
            "arms ({name: [overrides]}), seeds, eval.args, predict.images and predict.args. Relative paths are\n"
            "relative to the run file. Each stage calls the existing command in its own process and leaves a marker\n"
            "in RUN/.segpt_run when it finishes; a re-run skips finished stages and resumes at the failed one."
        ),
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=(
            "examples:\n"
            "  segpt-run run.yaml --dry-run                 # print the stage plan, run nothing\n"
            "  segpt-run run.yaml train.epochs=40           # extra Hydra overrides go to build and train\n"
            "  segpt-run run.yaml --force eval              # redo evaluation only (a stage kind or an id such as train-unet_s42)\n"
        ),
    )
    parser.add_argument("config", metavar="CONFIG", help="Run file (YAML)")
    parser.add_argument("overrides", nargs="*", metavar="KEY=VALUE", help="Hydra overrides added to build and train")
    parser.add_argument("--dry-run", action="store_true", help="Print the stage plan and the command of each stage; do no work")
    parser.add_argument("--force", action="append", default=[], metavar="STAGE",
                        help="Re-run this stage (build, train, eval, predict, or an id such as train-unet_s42); repeat the flag for more")
    return parser.parse_args(argv)

def segpt_run(argv=None) -> None:
    """Entry point for the `segpt-run` command.

    Run the whole chain from one run file, skipping finished stages.

    Example
    -------
        segpt-run run.yaml --dry-run
        segpt-run run.yaml --force eval
    """
    args = _run_main_args(argv)
    try:
        cfg = _load_run_file(args.config, args.overrides)
        run_pipeline(cfg, dry_run=args.dry_run, force=args.force)
    except RunError as e:
        print(f"[error] {e}", file=sys.stderr)
        sys.exit(1)

In [ ]:
# Verify helpers: a tiny raw dataset (6 images, one bright square each) and a run-file writer
def _make_raw(root, n=6, size=64):
    rng = np.random.default_rng(0)
    for sub in ('images', 'masks'): (Path(root) / sub).mkdir(parents=True, exist_ok=True)
    for i in range(n):
        im = rng.integers(20, 80, (size, size), dtype=np.uint8); m = np.zeros((size, size), np.uint8)
        y, x = rng.integers(8, 36, 2); im[y:y+20, x:x+20] = 220; m[y:y+20, x:x+20] = 255
        cv2.imwrite(str(Path(root) / 'images' / f's{i}.png'), im); cv2.imwrite(str(Path(root) / 'masks' / f's{i}.png'), m)
    return Path(root)

_TINY = ['patch.patch_size=32', 'patch.stride=32', 'patch.min_fg_pixels=1', 'patch.max_empty_frac=1.0', 'patch.num_workers=0',
         'model.source=custom_unet', '+model.custom_unet.encoder_depth=2', '+model.custom_unet.base_channels=8', '+model.custom_unet.num_groups=4',
         'train.crop_size=32', 'inference.tile_size=32', 'inference.overlap=8', '+data.class_names=[defect]', 'train.epochs=2', 'train.batch_size=4', 'train.device=cpu', 'train.use_amp=false', 'train.num_workers=0', 'train.freeze_epochs=0']

def _write_run_file(root, arms=None, seeds=(42,), images=True, repo=None):
    root = Path(root); raw = root / 'raw'
    if not raw.exists(): _make_raw(raw)
    cfg = {'run_dir': 'run', 'config': str(Path(_find_repo_root()) / 'configs' / 'train.yaml'),
           'overrides': [f'data.raw_dir={raw}', *_TINY], 'arms': arms or {'tiny': []}, 'seeds': list(seeds),
           'eval': {'args': ['--device', 'cpu']}}
    if images: cfg['predict'] = {'images': [str(raw / 'images')], 'args': ['--device', 'cpu']}
    p = root / 'run.yaml'; p.write_text(yaml.safe_dump(cfg, sort_keys=False)); return p

class _Spy:
    "Stand-in for the subprocess call: records stage ids, returns 0 unless the id is in `fail`."
    def __init__(self, fail=()): self.calls, self.fail = [], set(fail)
    def __call__(self, st):
        self.calls.append(st['id']); return 1 if st['id'] in self.fail else 0

p = _write_run_file(tempfile.mkdtemp(), arms={'a': [], 'b': ['model.arch=x']}, seeds=(1, 2))
cfg = _load_run_file(p)
[s['id'] for s in plan_run(cfg)]

In [ ]:
#| test
# the plan has every stage in order, with arms x seeds, and patch_dir defaults inside the run directory
ids = [s['id'] for s in plan_run(cfg)]
assert ids == ['build', 'train-a_s1', 'train-a_s2', 'train-b_s1', 'train-b_s2', 'eval',
               'predict-a_s1', 'predict-a_s2', 'predict-b_s1', 'predict-b_s2'], ids
stages = {s['id']: s for s in plan_run(cfg)}
assert f"data.patch_dir={Path(cfg['run_dir']) / 'patches'}" in stages['build']['args']
assert 'train.seed=2' in stages['train-a_s2']['args'] and 'model.arch=x' in stages['train-b_s1']['args']
assert all(a.endswith(('_s1', '_s2')) or '/train/' not in a for a in stages['eval']['args'] if a.startswith(cfg['run_dir']) and '/train/' in a)
assert _SEED_SUFFIX.sub('', 'unet_s42') == 'unet'    # the same grouping rule segpt-eval uses
# no predict images -> no predict stages; an explicit patch_dir is kept
cfg2 = _load_run_file(_write_run_file(tempfile.mkdtemp(), images=False))
assert not [s for s in plan_run(cfg2) if s['stage'] == 'predict']

In [ ]:
#| test
# --dry-run prints every stage and its command, runs nothing and creates nothing
import contextlib, io
tmp = Path(tempfile.mkdtemp()); p = _write_run_file(tmp, arms={'a': [], 'b': []})
buf = io.StringIO()
with contextlib.redirect_stdout(buf): segpt_run([str(p), '--dry-run'])
out = buf.getvalue()
for sid in ('build', 'train-a_s42', 'train-b_s42', 'eval', 'predict-a_s42', 'predict-b_s42'): assert sid in out, sid
assert 'segpt-train' in out and 'segpt-predict' in out
assert not (tmp / 'run').exists(), 'dry-run must not create the run directory'

In [ ]:
#| test
# markers: a re-run skips finished stages (marker files untouched); --force re-runs only the named stage
tmp = Path(tempfile.mkdtemp()); cfg = _load_run_file(_write_run_file(tmp, arms={'a': [], 'b': []}))
spy = _Spy(); r = run_pipeline(cfg, execute=spy)
assert spy.calls == [s['id'] for s in plan_run(cfg)] and set(r.values()) == {'done'}
mtimes = {m.name: m.stat().st_mtime_ns for m in (tmp / 'run' / '.segpt_run').iterdir()}
assert len(mtimes) == len(spy.calls) and (tmp / 'run' / 'run.yaml').exists()
spy2 = _Spy(); r = run_pipeline(cfg, execute=spy2)
assert spy2.calls == [] and set(r.values()) == {'skipped'}
assert mtimes == {m.name: m.stat().st_mtime_ns for m in (tmp / 'run' / '.segpt_run').iterdir()}
spy3 = _Spy(); run_pipeline(cfg, force=['eval'], execute=spy3); assert spy3.calls == ['eval'], spy3.calls
spy4 = _Spy(); run_pipeline(cfg, force=['train-b_s42'], execute=spy4); assert spy4.calls == ['train-b_s42']
spy5 = _Spy(); run_pipeline(cfg, force=['predict'], execute=spy5); assert spy5.calls == ['predict-a_s42', 'predict-b_s42']
try: run_pipeline(cfg, force=['nope'], execute=_Spy()); raise AssertionError('unknown stage accepted')
except RunError as e: assert 'nope' in str(e)

In [ ]:
#| test
# a failed stage stops the run with exit 1 and no marker; the next run resumes exactly there
tmp = Path(tempfile.mkdtemp()); p = _write_run_file(tmp, arms={'a': [], 'b': []}); cfg = _load_run_file(p)
spy = _Spy(fail=['train-b_s42'])
try: run_pipeline(cfg, execute=spy); raise AssertionError('failure was swallowed')
except RunError as e: assert 'train-b_s42' in str(e)
assert spy.calls == ['build', 'train-a_s42', 'train-b_s42'], spy.calls
markers = {m.stem for m in (tmp / 'run' / '.segpt_run').iterdir()}
assert markers == {'build', 'train-a_s42'}, markers
spy = _Spy(); run_pipeline(cfg, execute=spy)
assert spy.calls == ['train-b_s42', 'eval', 'predict-a_s42', 'predict-b_s42'], spy.calls
# the same failure through the command: message on stderr, exit 1, no traceback
bad = tmp / 'bad.yaml'; bad.write_text('run_dir: r\n')
err = io.StringIO()
with contextlib.redirect_stderr(err):
    try: segpt_run([str(bad)]); raise AssertionError('no exit')
    except SystemExit as e: assert e.code == 1
assert "[error]" in err.getvalue() and "'config' is required" in err.getvalue() and 'Traceback' not in err.getvalue()
# training that crashed mid-way resumes from latest.pth
d = Path(cfg['run_dir']) / 'train' / 'a_s42'; (d / 'checkpoints').mkdir(parents=True, exist_ok=True); (d / 'checkpoints' / 'latest.pth').write_bytes(b'x')
assert 'train.resume=true' in {s['id']: s for s in plan_run(cfg)}['train-a_s42']['args']

In [ ]:
#| test
# the real chain on tiny synthetic data (CPU): all outputs, a real failing train stage, resume, skip, forced stage
import os, sys
if sys.version_info >= (3, 14):
    print('skipped: Hydra (segpt-build / segpt-train) does not start on Python 3.14 yet (LazyCompletionHelp); run this on 3.12 or 3.13')
else:
    os.environ['CUDA_VISIBLE_DEVICES'] = ''    # CPU test: the stages' subprocesses must not touch (or wait for) a busy GPU
    tmp = Path(tempfile.mkdtemp()); p = _write_run_file(tmp)
    bad = yaml.safe_load(p.read_text()); bad['arms'] = {'tiny': ['model.source=does_not_exist']}
    badp = tmp / 'bad_run.yaml'; badp.write_text(yaml.safe_dump(bad, sort_keys=False))
    try: segpt_run([str(badp)]); raise AssertionError('a failing train stage must exit 1')
    except SystemExit as e: assert e.code == 1
    run = tmp / 'run'; mk = run / '.segpt_run'
    assert (mk / 'build.done').exists() and not (mk / 'train-tiny_s42.done').exists()
    build_mtime = (mk / 'build.done').stat().st_mtime_ns
    segpt_run([str(p)])     # fixed run file: build is skipped, training runs, the rest follows
    assert (mk / 'build.done').stat().st_mtime_ns == build_mtime
    train = run / 'train' / 'tiny_s42'
    assert (train / 'checkpoints' / 'best.pth').exists() and (train / '.hydra' / 'config.yaml').exists()
    assert (run / 'eval' / 'comparison.json').exists()
    pred = run / 'predict' / 'tiny_s42'
    assert (pred / 'summary.json').exists() and len(list((pred / 'masks').glob('*.png'))) == 6
    marks = {m.name: m.stat().st_mtime_ns for m in mk.iterdir()}
    segpt_run([str(p)])     # nothing left to do
    assert marks == {m.name: m.stat().st_mtime_ns for m in mk.iterdir()}
    before = (train / 'checkpoints' / 'best.pth').stat().st_mtime_ns
    segpt_run([str(p), '--force', 'eval'])     # only eval re-runs
    assert (mk / 'eval.done').stat().st_mtime_ns > marks['eval.done'] and (train / 'checkpoints' / 'best.pth').stat().st_mtime_ns == before
    assert (mk / 'train-tiny_s42.done').stat().st_mtime_ns == marks['train-tiny_s42.done']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()